# Aligning MSI and Visium HD data with `spatialwarp`

This notebook shows how to combine a mass spectrometry imaging (MSI) dataset with a Visium HD
dataset from the same tissue, so that every Visium spot ends up with both its gene
expression **and** its metabolite/lipid intensities.

**The problem:** MSI and Visium HD are usually scanned as two separate sections, each with its
own H&E image and its own coordinate system. Before you can compare them spot-by-spot, three
things need to happen:

1. **Fix the MSI grid**: MSI spot coordinates rarely line up with the MSI section's own H&E
   image out of the box (Step 1). If the MSI technology you use already align the MSI measurement to the H&E you can skip this step
2. **Align the two H&E images to each other**: the MSI section and the Visium section are two
   different scans, so they need to be registered to one another (Step 3).
3. **Merge**: once both are on the same "map", the nearest MSI spot for every Visium spot is
   found automatically and the two datasets are merged (end of Step 3).

In [3]:
import sys
sys.path.insert(0, "msi")  # examples/msi/ contains msi_loader.py, build_msi_spatialdata.py

from build_msi_spatialdata import build_msi_spatialdata
import spatialdata_io
import spatialwarp

/home/croizer/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/croizer/.local/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


A couple of steps below open a window where you click points with your mouse. For that to
work in Jupyter, run this cell first to switch to a real window backend instead of the default
static-image mode (`tk` works out of the box with no extra install; use `qt` instead if you
have PyQt5/PySide installed and prefer it).

In [ ]:
%matplotlib tk

## Step 1   Place the MSI data on its own image

MSI instruments record each spot's coordinates in their own internal grid, which almost never
matches the pixel coordinates of the section's own H&E image. This step fixes that:

1. It turns the MSI spots into a small preview image from their raw intensities.
2. A window opens showing that preview next to the real H&E image, click a point on the left,
   then click the matching point on the right, for about 5-10 pairs (pick features you can
   recognize on both, e.g. tissue edges or clear intensity blobs).
3. Close the window when you're done. The package uses your clicks to line up every MSI spot
   with the H&E image automatically.

In [5]:
msi_he_image = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/H&E - consecutive seciton/L12_E_20x_Section5_GENIAL.tiff"
    
metabo_intensity_csv ="/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/L12-E/20251001-Brosch-Sample02-Total Ion Count.csv"
metabo_annotation_csv = '/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/Metabolomics list.csv'
metabo_region_csv =  "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/L12-E/20251001-Brosch-Sample02_regionspots.csv"

msi_metabo_sdata = build_msi_spatialdata(
    image_path=msi_he_image,
    intensity_csv=metabo_intensity_csv,
    annotation_csv=metabo_annotation_csv,
    region_csv=metabo_region_csv,
    aligned_coords_csv="transformed_coordinates_L12_metabo.csv",
)

RuntimeError: Exception thrown in SimpleITK LandmarkBasedTransformInitializer: /tmp/SimpleITK-build/ITK-prefix/include/ITK-5.4/itkLandmarkBasedTransformInitializer.hxx:208:
ITK ERROR: LandmarkBasedTransformInitializer(0x7f7cac02a1b0):  insufficient number of landmarks, expected 3 got 1

**Check it worked:** the plot below overlays the (now-aligned) MSI spots on the H&E image.
Drag the alpha slider: the colored spots should trace out real tissue features in the image,
not float off to one side.

In [ ]:
import tifffile
from spatialwarp.qc import plot_overlay

msi_metabo_table = msi_metabo_sdata.tables["msi"]
plot_overlay(
    image=tifffile.imread(msi_he_image),
    points_xy=msi_metabo_table.obsm["spatial"],
    values=msi_metabo_table.X.sum(axis=1),
)

In [4]:
msi_metabo_sdata

SpatialData object
├── Images
│     └── 'he': DataArray[cyx] (3, 7048, 8192)
└── Tables
      └── 'msi': AnnData (88707, 77)
with coordinate systems:
    ▸ 'global', with elements:
        he (Images)

## Step 2   Load the Visium HD data

Visium's own pipeline (Space Ranger) already guarantees that each bin's coordinates match its
H&E image pixel-for-pixel, so there's nothing to fix here, we just load it.

In [5]:
visium_binned_outputs = "/home/croizer/Documents/04_Collaborations/03_Genial/01_Mario/GENIAL_VisiumHD_LAB5954/Reprocessed/L12/outs/binned_outputs/square_016um"
visium_he_image = "/home/croizer/Documents/04_Collaborations/03_Genial/01_Mario/GENIAL_VisiumHD_LAB5954/20250925_HE_VisiumHD_slide/Slide1_A_L12_Edge.tiff"

# fullres_image_file is required: without it, visium_hd() only loads the downscaled
# hires/lowres preview images, which are NOT in the same pixel space as obsm['spatial']
# (verified: with fullres_image_file set, obsm['spatial'] min/max match the raw
# tissue_positions.parquet pxl_row_in_fullres/pxl_col_in_fullres exactly).
# bin_size=16 narrows the tables/shapes to just square_016um (images are unaffected by this).
visium_sdata = spatialdata_io.visium_hd(
    visium_binned_outputs,
    dataset_id="L12",
    bin_size=16,
    fullres_image_file=visium_he_image,
)
visium_sdata

/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


SpatialData object
├── Images
│     ├── 'L12_full_image': DataTree[cyx] (3, 8192, 8006), (3, 4096, 4003), (3, 2048, 2001), (3, 1024, 1000), (3, 512, 500)
│     ├── 'L12_hires_image': DataArray[cyx] (3, 6000, 5864)
│     └── 'L12_lowres_image': DataArray[cyx] (3, 600, 587)
├── Shapes
│     └── 'L12_square_016um': GeoDataFrame shape: (142954, 1) (2D shapes)
└── Tables
      └── 'square_016um': AnnData (142954, 18085)
with coordinate systems:
    ▸ 'downscaled_hires', with elements:
        L12_hires_image (Images), L12_square_016um (Shapes)
    ▸ 'downscaled_lowres', with elements:
        L12_lowres_image (Images), L12_square_016um (Shapes)
    ▸ 'global', with elements:
        L12_full_image (Images), L12_hires_image (Images), L12_lowres_image (Images), L12_square_016um (Shapes)

## Step 3  Align the two H&E images, then merge

Even though each dataset's own points are now correctly placed on its own image (Steps 1-2),
the MSI section and the Visium section are still two separate scans, their images don't line
up with *each other* yet. This step:

1. Opens the two H&E images side by side. Click a handful of matching points (the same tissue
   landmark on both sides), then close the window.
2. Registers the two images from those points (takes a few seconds to minutes depending on scan size).
3. Uses that registration to find, for every Visium spot, its nearest MSI spot, and merges the
   matched metabolite intensities into the Visium data.

This registration only depends on the two H&E images, not on which MSI analyte you're using,
so if you have more than one MSI dataset for this slide (e.g. metabolomics *and* lipidomics),
you only need to do this step **once** and reuse the result (see Step 4).

In [6]:
import tifffile
from spatialwarp.landmark_picker import pick_landmarks
from spatialwarp.registration import register_elastic

msi_he_array = tifffile.imread(msi_he_image)
visium_he_array = tifffile.imread(visium_he_image)

# Click a point on the left (MSI) image, then its match on the right (Visium) image;
# repeat for ~5-10 well-spread landmarks, then close the window.
moving_landmarks, fixed_landmarks = pick_landmarks(
    msi_he_array, visium_he_array, output_csv="landmarks_L12.csv"
)


In [7]:

registration_result = register_elastic(
    moving_image=msi_he_array,
    fixed_image=visium_he_array,
    moving_landmarks=moving_landmarks,
    fixed_landmarks=fixed_landmarks,
    mesh_size=(8, 8),
    number_of_iterations=100,
)
registration_result.save("registration_L12")

In [ ]:
adata_metabo = spatialwarp.align(
    moving=msi_metabo_sdata,
    fixed=visium_sdata,
    registration_result=registration_result,
    distance_threshold=20.0,
    fixed_image_key="L12_full_image",
    fixed_table_key="square_016um",
    moving_obsm_key="metabolite",
)
adata_metabo

`adata_metabo` is now a regular Visium `AnnData` object with gene expression as usual, plus the
matched metabolite intensities in `adata_metabo.obsm["metabolite"]`. The cells below are just a
few example plots to sanity-check the result; feel free to skip straight to Step 4 if you don't
need them.

In [9]:
import scanpy as sc
sc.pp.calculate_qc_metrics(adata_metabo, inplace=True, log1p=True)

In [ ]:
import matplotlib.pyplot as plt

# metabolite values now live in adata_metabo.obsm["metabolite"] (a DataFrame), not .obs,
# so plot them directly rather than via sc.pl.spatial's color= (which only looks in .obs/.var)
metabolite_col = "4-Hydroxyphenylpyruvate*"
values = adata_metabo.obsm["metabolite"][metabolite_col].values
coords = adata_metabo.obsm["spatial"]

fig, ax = plt.subplots(figsize=(8, 8))
sca = ax.scatter(coords[:, 0], -coords[:, 1], c=values, cmap="viridis", s=4)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(metabolite_col)
plt.colorbar(sca, ax=ax, shrink=0.7)
plt.show()

In [11]:
import scanpy as sc
sc.pl.spatial(
    adata_metabo, color='total_counts', spot_size=20)

/tmp/ipykernel_660851/408030824.py:2: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# map each channel to an RGB color; metabolite columns live in obsm["metabolite"],
# while total_counts (from sc.pp.calculate_qc_metrics) stays in .obs
channels = {
    "4-Hydroxyphenylpyruvate*": ("red", adata_metabo.obsm["metabolite"]["4-Hydroxyphenylpyruvate*"].values),
    "total_counts": ("green", adata_metabo.obs["total_counts"].values),
}
color_idx = {"red": 0, "green": 1, "blue": 2}

spatial_coords = adata_metabo.obsm["spatial"]
rgb_image = np.zeros((adata_metabo.n_obs, 3))

for label, (color, values) in channels.items():
    values = values.astype(float)
    vmax = np.nanpercentile(values, 99)
    normalized = np.clip(values / vmax, 0, 1) if vmax > 0 else values
    rgb_image[:, color_idx[color]] = normalized

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(spatial_coords[:, 0], -spatial_coords[:, 1], c=rgb_image, s=2, alpha=0.9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("IF-like overlay: " + " vs ".join(channels.keys()))

legend_elements = [Patch(facecolor=color, label=label) for label, (color, _) in channels.items()]
ax.legend(handles=legend_elements, loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

In [12]:
adata_metabo

AnnData object with n_obs × n_vars = 98842 × 18085
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'nearest_index', 'nearest_distance', 'Taurine', 'Imidazole-4-acetate / Thymine', 'Dihydrothymine', 'Pyrroline-hydroxy-carboxylate / Oxoproline', 'Aspartate', 'Malate*', 'Adenine', 'Hypoxanthine # Threonate*', 'Ethanolamine phosphate', 'Glutamine*', 'Glutamate*', 'Quinolinate', 'Phosphoenolpyruvate (PEP)', 'Urate', 'Glycerol 3-phosphate', '4-Hydroxyphenylpyruvate*', 'Deoxyribose phosphate', 'Glycerophosphoethanolamine', 'FA 14:0 (Myristate)', 'Asparaginyl-Proline', 'Methionyl-Serine', 'Cytidine', 'Glutamyl-Taurine', 'FA 16:1 Palmiteoleate-Sapienate', 'Threoninyl-Histidine', 'FA 16:0 Palmitate', 'Glucose phosphate / Fructose phosphate*', 'Asparaginyl-Methionine', 'Glycerate diphosphate', 'FA 17:1 Heptadecenoate', 'FA 17:0 Heptadecanoate', 'Glutaminyl-Glutamate', 'Glutamyl-Glutamate', 'FA 18:3 Linolenate', 'FA 18:2 Linoleate', 'FA 18:1 Oleate', 'FA 18:0 Stearate', 'O

## Step 4 Add a second MSI dataset (lipidomics) into the same object

Same as Step 1, but for the lipidomics export, you'll click landmarks again to place the
lipid grid on the H&E image. Registration itself is skipped this time: we just reuse
`registration_result` from Step 3, since it only depends on the two H&E images, not the analyte.

The matched lipid values are merged straight
into `adata_metabo.obsm["lipid"]`, so you end up with **one** object holding gene expression,
metabolites, and lipids together.

In [ ]:
lipid_intensity_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/L12-E/20251002_Brosch_Lipids_Neg2_1-Total Ion Count.csv"
lipid_annotation_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/Final-Planque-Lip-Oct25.csv"
lipid_region_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/L12-E/20251002_Brosch_Lipids_Neg3_regionspots.csv"

msi_lipid_sdata = build_msi_spatialdata(
    image_path=msi_he_image,
    intensity_csv=lipid_intensity_csv,
    annotation_csv=lipid_annotation_csv,
    region_csv=lipid_region_csv,
    aligned_coords_csv="transformed_coordinates_L12_lipid.csv",
)

lipid_matched = spatialwarp.align(
    moving=msi_lipid_sdata,
    fixed=visium_sdata,
    registration_result=registration_result,
    distance_threshold=20.0,
    fixed_image_key="L12_full_image",
    fixed_table_key="square_016um",
    moving_obsm_key="lipid",
)

# lipid_matched may keep a slightly different set of spots than adata_metabo (each analyte's
# own MSI grid can have different coverage/gaps) -- reindex onto adata_metabo's spots so both
# analytes live in one object, filling any gap in lipid coverage with NaN.
adata_metabo.obsm["lipid"] = lipid_matched.obsm["lipid"].reindex(adata_metabo.obs_names)
adata_metabo.obs["nearest_distance_lipid"] = lipid_matched.obs["nearest_distance"].reindex(adata_metabo.obs_names)
adata_metabo

`adata_metabo` is now a regular Visium `AnnData` object holding gene expression, plus the
matched metabolite and lipid intensities in `.obsm["metabolite"]` / `.obsm["lipid"]`.


The plot below is a final visual sanity check: it shows where each Visium point landed once
warped into MSI space. It should trace out the same tissue shape as the H&E image, not a
scattered cloud.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(*adata_metabo.obsm["spatial_warped"].T, s=1, alpha=0.3)
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("Visium points warped into MSI space, kept after distance filter")
plt.show()